# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The metadata object provides dataset information
metadata = dataset.metadata # This is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id` values.

In [ ]:
# List RecordSets contained in the dataset
print("Available record sets and their field @ids:")

record_sets = metadata.record_sets()
for rec in record_sets:
    print(f"  Record Set: {rec['@id']}")
    # List the fields
    if 'field' in rec:
        for f in rec['field']:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            print(f"    Field: {field_id}")

## 3. Data Extraction
Load data from the main record set(s) into DataFrames for analysis.

**Note:** Always refer to record sets, fields, and columns **by their `@id`**.

In [ ]:
# Gather the record set @ids
main_record_set_ids = [rec['@id'] for rec in metadata.record_sets()]
print("Record sets found:", main_record_set_ids)

// For this FAIR^2 dataset, let's load the first record set for demonstration.
dfs = {}  # Map from record_set @id to DataFrame
for rec_id in main_record_set_ids:
    # Load all records as list of dicts
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dfs[rec_id] = df
    print(f"DataFrame created for record set {rec_id} with shape {df.shape}")

# Show head and column info for the first record set
main_rs = main_record_set_ids[0]
print(f"Columns: {dfs[main_rs].columns.tolist()}")
dfs[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some data processing steps:
- Filter the DataFrame by a numeric field (for example, 'cr:coveragePercent' if present).
- Normalize numeric values.
- Group by a categorical field (such as a protein type or description).

All fields and columns are referenced by their `@id`, as required.

In [ ]:
# Select the main DataFrame
df = dfs[main_rs]

# Inspect available columns
print("First 10 columns:", df.columns[:10].tolist())

# For EDA, try to find a common numeric field
possible_numeric_fields = [c for c in df.columns if 'coverage' in c or 'percent' in c or 'MW' in c or 'abundance' in c or 'count' in c or 'score' in c]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]

# Try to convert column to numeric, ignoring errors (e.g. in case values are str)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].mean()  # For demo, filter above average
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field} > mean ({threshold:.2f})")
display_cols = [c for c in [numeric_field] if c in filtered_df.columns]
print(filtered_df[display_cols].head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Group by a categorical field, e.g., description or accession
possible_group_fields = [c for c in df.columns if 'description' in c or 'accession' in c or 'id' in c]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by '{group_field}':")
    print(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version.

In [ ]:
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(10, 4))
    plt.hist(df[numeric_field].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        plt.hist(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
        plt.title(f"Distribution of Normalized {numeric_field}")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()

## 6. Conclusion

- This notebook illustrated how to explore a FAIR^2 Croissant-conformant dataset using `mlcroissant`.
- Record sets, fields, and columns were referenced by their `@id` as per best practices.
- Basic EDA and visualization steps were performed on mass spectrometry records.

This provides a reproducible template for working with Croissant-schema datasets for your own research and analysis needs.